In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "aryan208/financial-transactions-dataset-for-fraud-detection"
)

print(path)

In [ ]:
import os

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import pandas as pd

csv_file = (
    path
    + "/financial_fraud_detection_dataset.csv"
)

df = pd.read_csv(csv_file)

print(df.shape)

print(df.columns.tolist())

df.head()

In [ ]:
print(df["is_fraud"].value_counts())

print(
    df["is_fraud"]
      .value_counts(normalize=True)
)

In [ ]:
import pandas as pd
import requests
from datetime import datetime

# Solana Mainnet Public RPC Endpoint
url = "https://api.mainnet-beta.solana.com"
headers = {"Content-Type": "application/json"}

payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "getBlock",
    "params": [
        280000000,
        {
            "encoding": "json",
            "maxSupportedTransactionVersion": 0,
            "transactionDetails": "full",
            "rewards": False
        }
    ]
}

try:
    resp = requests.post(url, json=payload, headers=headers)
    resp.raise_for_status()
    block_data = resp.json()

    if "result" in block_data and block_data["result"] is not None:
        transactions = block_data["result"].get("transactions", [])
        block_time = block_data["result"].get("blockTime")

        # Map blockTime to a timestamp object
        dt_timestamp = pd.to_datetime(block_time, unit='s') if block_time else pd.Timestamp.now()

        parsed_txs = []
        for tx in transactions:
            # Get signatures and account keys
            sig = tx["transaction"]["signatures"][0] if tx["transaction"]["signatures"] else None
            account_keys = tx["transaction"]["message"].get("accountKeys", [])

            # Map Solana accounts
            sender = account_keys[0] if len(account_keys) > 0 else "Unknown"
            receiver = account_keys[1] if len(account_keys) > 1 else "Unknown"

            # Check for errors to set is_fraud flag
            meta = tx.get("meta")
            err = meta.get("err") if meta else None
            is_fraud = err is not None

            # Solana lamports to SOL conversion for amount
            fee = meta.get("fee", 0) if meta else 0
            amount_sol = fee / 10**9

            tx_info = {
                "transaction_id": sig,
                "timestamp": dt_timestamp,
                "sender_account": sender,
                "receiver_account": receiver,
                "amount": amount_sol,
                "transaction_type": "transfer",
                "location": "Solana Network",
                "is_fraud": is_fraud
            }
            parsed_txs.append(tx_info)

        solana_df = pd.DataFrame(parsed_txs)
    else:
        raise ValueError("No block data returned or slot has been pruned.")
except Exception as e:
    print(f"Could not fetch live block data: {e}")
    print("Falling back to schema-compliant local mock data.")
    solana_df = pd.DataFrame([
        {
            "transaction_id": "3A7d8F9g...",
            "timestamp": pd.Timestamp.now(),
            "sender_account": "3uv7g8...",
            "receiver_account": "8vY1b9...",
            "amount": 0.05,
            "transaction_type": "transfer",
            "location": "Solana Network",
            "is_fraud": False
        },
        {
            "transaction_id": "8Hj2Kl9s...",
            "timestamp": pd.Timestamp.now(),
            "sender_account": "9xR4f1...",
            "receiver_account": "3uv7g8...",
            "amount": 0.01,
            "transaction_type": "transfer",
            "location": "Solana Network",
            "is_fraud": True
        }
    ])

solana_df.head(10)

In [ ]:
# Check the value distribution of 'transaction_type' in solana_df
print(solana_df['transaction_type'].value_counts(dropna=False))

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["is_fraud"],
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.10,
    stratify=train_df["is_fraud"],
    random_state=42
)

In [ ]:
chunks = pd.read_csv(
    csv_file,
    chunksize=100000
)

In [ ]:
import torch
import torch.nn as nn

class FraudNet(nn.Module):

    def __init__(
        self,
        input_dim
    ):
        super().__init__()

        self.model = nn.Sequential(

            nn.Linear(input_dim,512),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(512,256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128,1)

        )

    def forward(
        self,
        x
    ):
        return self.model(x)

In [ ]:
class FocalLoss(nn.Module):

    def __init__(
        self,
        alpha=10,
        gamma=2
    ):
        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(
        self,
        logits,
        targets
    ):

        bce = nn.functional.binary_cross_entropy_with_logits(
            logits.squeeze(),
            targets.float(),
            reduction="none"
        )

        pt = torch.exp(-bce)

        return (
            self.alpha
            * ((1 - pt) ** self.gamma)
            * bce
        ).mean()

In [ ]:
best_loss = float("inf")

patience = 5

stagnant = 0

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Identify numerical columns for scaling
features_cols = [
    'amount',
    'time_since_last_transaction',
    'spending_deviation_score',
    'velocity_score',
    'geo_anomaly_score'
]

# Fill missing values for tabular columns
for col in features_cols:
    mean_val = train_df[col].mean()
    train_df[col] = train_df[col].fillna(mean_val)
    val_df[col] = val_df[col].fillna(mean_val)
    test_df[col] = test_df[col].fillna(mean_val)

# Extract numpy arrays
X_train = train_df[features_cols].values
y_train = train_df['is_fraud'].astype(float).values

X_val = val_df[features_cols].values
y_val = val_df['is_fraud'].astype(float).values

# Scale our numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Convert to PyTorch tensors
import torch.utils.data as data_utils

train_dataset = data_utils.TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

val_dataset = data_utils.TensorDataset(
    torch.tensor(X_val_scaled, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32)
)

train_loader = data_utils.DataLoader(train_dataset, batch_size=2048, shuffle=True)
val_loader = data_utils.DataLoader(val_dataset, batch_size=2048, shuffle=False)

In [ ]:
# Determine input dimensions and instantiate the model
input_dim = X_train_scaled.shape[1]
model = FraudNet(input_dim=input_dim)

# Setup optimization and criterion
criterion = FocalLoss(alpha=10, gamma=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)

In [ ]:
# Training Loop
epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Training model on {device}...")
for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * len(batch_x)

    avg_train_loss = total_train_loss / len(train_dataset)

    # Validation Phase
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_val_loss += loss.item() * len(batch_x)

    avg_val_loss = total_val_loss / len(val_dataset)

    print(f"Epoch {epoch}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # Save checkpoint if loss improves
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        stagnant = 0
        torch.save(model.state_dict(), "fraud_model.pt")
        print("==> Saved checkpoint with lower validation loss!")
    else:
        stagnant += 1
        if stagnant >= patience:
            print("Early stopping triggered!")
            break

Training model on cpu...
Epoch 1/10 | Train Loss: 0.4538 | Val Loss: 0.4331
==> Saved checkpoint with lower validation loss!
Epoch 2/10 | Train Loss: 0.4322 | Val Loss: 0.4255
==> Saved checkpoint with lower validation loss!
Epoch 3/10 | Train Loss: 0.4275 | Val Loss: 0.4235
==> Saved checkpoint with lower validation loss!
Epoch 4/10 | Train Loss: 0.4254 | Val Loss: 0.4226
==> Saved checkpoint with lower validation loss!
Epoch 5/10 | Train Loss: 0.4242 | Val Loss: 0.4223
==> Saved checkpoint with lower validation loss!
Epoch 6/10 | Train Loss: 0.4235 | Val Loss: 0.4218
==> Saved checkpoint with lower validation loss!
Epoch 7/10 | Train Loss: 0.4229 | Val Loss: 0.4217
==> Saved checkpoint with lower validation loss!
Epoch 8/10 | Train Loss: 0.4227 | Val Loss: 0.4215
==> Saved checkpoint with lower validation loss!
Epoch 9/10 | Train Loss: 0.4224 | Val Loss: 0.4218


### Evaluation & Solana Analysis
Now that the model is trained, let's load the best model checkpoint and apply it to predict fraud on the live Solana transaction dataset (`solana_df`).

In [ ]:
# Load the best model weights
model.load_state_dict(torch.load("fraud_model.pt", map_location=device))
model.eval()

# Prepare the Solana features for prediction
# Let's align Solana features with the ones our model expects.
# Since solana_df lacks columns like 'time_since_last_transaction', we'll default them to 0 or dataset averages.
solana_features = pd.DataFrame(index=solana_df.index)
solana_features['amount'] = solana_df['amount']
solana_features['time_since_last_transaction'] = 0.0
solana_features['spending_deviation_score'] = 0.0
solana_features['velocity_score'] = 0.0
solana_features['geo_anomaly_score'] = 0.0

# Scale Solana features using the exact same scaler used during training
X_solana_scaled = scaler.transform(solana_features[features_cols].values)

# Run prediction
with torch.no_grad():
    solana_tensors = torch.tensor(X_solana_scaled, dtype=torch.float32).to(device)
    logits = model(solana_tensors).squeeze()
    probabilities = torch.sigmoid(logits).cpu().numpy()

solana_df['fraud_probability'] = probabilities
solana_df['predicted_fraud'] = probabilities > 0.5

# Display the top records with high fraud probabilities
display(solana_df.sort_values(by='fraud_probability', ascending=False).head(10))

In [ ]:
torch.save(
    model.state_dict(),
    "fraud_model.pt"
)